# Kwartaal-pro-forma resultatenrekening met PROC COMPUTAB

## Samenvatting voor het management

Deze notebook bouwt de kwartaal-pro-forma resultatenrekening van een regionale bank rechtstreeks op uit maandelijkse grootboekgegevens met PROC COMPUTAB, de rapportagetabelprocedure van SAS/ETS. De procedure leidt de rentebaten, rentelasten, provisiebaten en bedrijfskosten van elke maand naar de juiste kalenderkwartaalkolom, gebruikt vervolgens rijprogrammeerblokken om de netto rentebaten, het resultaat voor belastingen en het nettoresultaat te berekenen, en een kolomblok om de vier kwartalen op te tellen tot een jaartotaal.

## Gegevensbronnen

De enkele synthetische dataset `bank_ledger` simuleert één boekjaar aan maandelijkse posten uit de resultatenrekening voor een middelgrote lokale bank. Twaalf maandelijkse waarnemingen worden inline gegenereerd met `call streaminit`/`rand`, zodat de notebook volledig zelfstandig werkt.

| Variabele | Type | Beschrijving |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Statementdatum per maandeinde (jan–dec) |
| `loanint`  | Num | Rente en provisies verdiend op de leningenportefeuille (USD duizenden) |
| `secint`   | Num | Rente verdiend op de beleggingsportefeuille (USD duizenden) |
| `depint`   | Num | Rente betaald op deposito's (USD duizenden) |
| `borrint`  | Num | Rente betaald op leningen / FHLB-voorschotten (USD duizenden) |
| `feeinc`   | Num | Niet-rentebaten (provisies en servicekosten) (USD duizenden) |
| `salaries` | Num | Salarissen en personeelskosten (USD duizenden) |
| `occupancy`| Num | Huisvestings- en apparatuurkosten (USD duizenden) |
| `othropex` | Num | Overige niet-rentegerelateerde bedrijfskosten (USD duizenden) |
| `provision`| Num | Voorziening voor kredietverliezen (USD duizenden) |
| `taxrate`  | Num | Effectief belastingtarief toegepast op het resultaat voor belastingen |

# Kwartaal-pro-forma resultatenrekening met PROC COMPUTAB

Financiële teams van banken zetten een maandelijks grootboek routinematig om in een **kwartaalresultatenrekening** die de netto rentebaten en het uiteindelijke nettoresultaat toont. `PROC COMPUTAB` (SAS/ETS) is hier speciaal voor gemaakt: het legt een programmeerbare tabel neer waarvan de **kolommen** de rapportageperioden zijn en de **rijen** de posten van de resultatenrekening, en het laat je subtotalen berekenen met gewone DATA step-logica in rij- en kolomblokken.

In deze notebook doen we het volgende:

1. Eén boekjaar aan synthetische maandelijkse grootboekgegevens genereren voor een lokale bank.
2. Elke maand naar het juiste kalenderkwartaal leiden en een kwartaalresultatenrekening opbouwen.
3. De netto rentebaten, het resultaat voor belastingen en het nettoresultaat berekenen in een **rijblok**, en de kwartalen optellen tot een jaartotaal in een **kolomblok**.
4. De `out=`-tabel hergebruiken zodat het berekende statement kan doorstromen naar vervolgrapportages.

## Stap 1 — Synthetische maandelijkse grootboekgegevens genereren

We simuleren twaalf waarnemingen per maandeinde. De rentebaten op leningen en effecten stijgen gedurende het jaar licht, de kosten van deposito's en leningen schalen mee met de renteomgeving, en de provisiebaten, bedrijfskosten en de voorziening voor kredietverliezen dragen realistische ruis van maand tot maand. `call streaminit` legt de seed vast zodat de gegevens reproduceerbaar zijn.

In [1]:
GEGEVENS bank_ledger;
   CALL streaminit(20240115);
   OPMAAK stmtdate date9.;
   DOE month = 1 TOT 12;
      /* Month-end statement date for fiscal year 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mild upward drift across the year + monthly noise */
      drift = 1 + 0.012 * (month - 1);

      /* Interest income (USD thousands) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Interest expense (USD thousands) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Non-interest income and expense (USD thousands) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provision for credit losses, occasionally elevated */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effective tax rate */
      taxrate = 0.21;

      UITVOER;
   EINDE;
   BEWAREN stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=bank_ledger noobs label;
   TITEL 'Synthetic Monthly Ledger — Community Bank FY2024';
   OPMAAK loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
UITVOEREN;

                                    Synthetic Monthly Ledger — Community Bank FY2024                                    

 STMTDATE   LOANINT  SECINT  DEPINT  BORRINT  FEEINC  SALARIES  OCCUPANCY  OTHROPEX  PROVISION  TAXRATE
28JAN2024  1,772.95  423.79  531.47   128.99  339.33    699.38     171.95    202.43     130.93     0.21
28FEB2024  1,824.38  421.13  564.85   125.53  294.09    767.29     162.79    307.61     123.25     0.21
28MAR2024  1,931.98  442.06  536.64   131.71  295.72    724.03     153.26    254.21     115.76     0.21
28APR2024  1,934.99  439.29  535.94   140.01  294.56    729.47     174.19    237.43     198.05     0.21
28MAY2024  1,949.31  447.35  591.44   124.42  299.78    739.13     165.08    223.32     105.57     0.21
28JUN2024  1,934.36  448.28  551.70   147.64  335.81    718.90     171.91    236.94     130.13     0.21
28JUL2024  1,936.57  439.22  565.70   133.82  293.21    743.87     174.16    247.19     199.20     0.21
28AUG2024  1,979.73  457.42  558.54   144.45  

## Stap 2 — De kwartaalresultatenrekening opbouwen

De kern van het rapport is de `PROC COMPUTAB`-stap hieronder.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** definieert vier kwartaalkolommen plus een jaartotaalkolom.
- Het ongelabelde **input-blok** stelt de automatische variabele **`_col_`** in met `qtr(stmtdate)`, waardoor elke maandelijkse waarneming naar de juiste kwartaalkolom wordt geleid. Omdat de input standaard getransponeerd wordt, komt elke grootboekvariabele terecht op de rij die dezelfde naam draagt.
- Het **rijblok** `inc_stmt:` draait één keer per kolom en berekent de afgeleide regels — netto rentebaten, totale niet-rentekosten, resultaat voor belastingen en nettoresultaat — precies zoals een accountant dat zou doen.
- Het **kolomblok** `total:` draait één keer per rij en telt de vier kwartalen op in de `fy`-kolom (jaartotaal).

De `rows ... / ...`-statements voegen titels, inspringing en lijnen toe (`ol` bovenlijn, `ul` onderlijn, `dul` dubbele onderlijn) zodat de output leest als een echte resultatenrekening.

In [2]:
TITEL 'Pro Forma Income Statement — Community Bancorp, Inc.';
title2 'Fiscal Year 2024  (Amounts in USD Thousands)';

PROCEDURE computab GEGEVENS=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Four quarters plus a rolled-up full-year column */
   columns qtr1 qtr2 qtr3 qtr4 fy / OPMAAK=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Full' 'Year' +3;

   /* Income statement rows, top to bottom */
   rows loanint  / 'Interest & Fees on Loans';
   rows secint   / 'Interest on Securities'        ul;
   rows intinc   / 'Total Interest Income';
   rows depint   / 'Interest on Deposits';
   rows borrint  / 'Interest on Borrowings'        ul;
   rows intexp   / 'Total Interest Expense';
   rows nii      / 'Net Interest Income'           ol skip;
   rows provision/ 'Provision for Credit Losses'   ul;
   rows niiap    / 'Net Int. Income after Prov.'   skip;
   rows feeinc   / 'Non-Interest Income'           ul;
   rows salaries / 'Salaries & Benefits';
   rows occupancy/ 'Occupancy & Equipment';
   rows othropex / 'Other Operating Expense'       ul;
   rows nonintexp/ 'Total Non-Interest Expense'    skip;
   rows pretax   / 'Pre-Tax Income'                ol;
   rows taxexp   / 'Income Tax Expense'            ul;
   rows netinc   / 'Net Income'                    dul;

   /* Input block: route each month to its calendar quarter */
   _col_ = qtr(stmtdate);

   /* Row block: compute statement subtotals across every column */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Column block: roll the four quarters into the full-year column */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
UITVOEREN;

                                  Pro Forma Income Statement — Community Bancorp, Inc.                                  
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Full  
                                                                               Year  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Interest & Fees on Loans
  loanint               5529.31      5818.66      5963.46      6276.96     23588.39  
  Interest on Securities
  secint                1286.98      1334.92      1342.03      1452.88      5416.81  
                    -----------  -----------  -----------  -----------  -----------  
  Total Interest Inc

## Stap 3 — De COMPUTAB-outputdataset hergebruiken

De `PROC COMPUTAB`-stap hierboven schreef zijn berekende tabel via `out=` naar `qtr_income`. Elke rij van die dataset is een post uit de resultatenrekening en elke kolomvariabele (`qtr1`–`qtr4`, `fy`) bevat de berekende waarde, zodat deze kan doorstromen naar vervolgrapportages. Hieronder drukken we de opgetelde jaartotaalkolom af om te bevestigen dat de cijfers kloppen.

In [3]:
TITEL 'COMPUTAB Output Dataset — Full-Year Column';

PROCEDURE AFDRUKKEN GEGEVENS=qtr_income label noobs;
   VARIABELE _row_ fy;
   OPMAAK fy comma12.1;
   label _row_ = 'Line Item' fy = 'Full Year';
UITVOEREN;

TITEL;

                                       COMPUTAB Output Dataset — Full-Year Column                                       
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


Line Item  Full Year
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6

NOTE: Option TITLE changed to COMPUTAB Output Dataset — Full-Year Column.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## De resultaten interpreteren

De pro-forma resultatenrekening leest van boven naar beneden als het regulatoire call report van een bank: totale rentebaten min totale rentelasten levert de **netto rentebaten** op — hier ongeveer \$20,5 miljoen voor het jaar — de belangrijkste winstmotor van de bank. Aftrekken van de **voorziening voor kredietverliezen**, optellen van de **provisiebaten** en verrekenen van de **niet-rentekosten** geeft een resultaat voor belastingen van ongeveer \$9,0 miljoen, en het toepassen van het effectieve belastingtarief van 21% levert een **nettoresultaat** van bijna \$7,1 miljoen op. Het kolomblok `total:` telt de vier kwartalen op in de jaartotaalkolom, zodat de jaartotalen per constructie aansluiten op de som van de kwartalen — bevestigd in de `out=`-dataset, waar het jaartotaal `netinc` van 7.081,6 gelijk is aan de som van de vier kwartaalcijfers.

Omdat alles wordt berekend binnen de programmeerbare rij- en kolomblokken van `PROC COMPUTAB`, vereist het vervangen door een echt maandelijks grootboek geen enkele wijziging aan de rapportagelogica — alleen de inputdataset verandert. De `out=`-dataset stelt het berekende statement vervolgens beschikbaar voor dashboards, trendanalyses of automatisering van bestuursstukken.